## StreamVault Data Profiling and Exploration

This notebook explores the StreamVault movie dataset, identifies data-quality issues and examines patterns that inform the licensing analysis.

### 1. Candidate List Inspection

Inspect the original candidate list for missing, duplicate and invalid TMDb IDs before preparing the working extraction list.

In [5]:
import pandas as pd
from pathlib import Path

# Locate the project root by finding the data folder
current_directory = Path.cwd()

project_root = current_directory

while not (project_root / "data" / "id_list.csv").exists():
    if project_root.parent == project_root:
        raise FileNotFoundError(
            "Could not locate data/id_list.csv. "
            "Check that the notebook is inside the project folder."
        )
    project_root = project_root.parent

# Load the original candidate list
file_path = project_root / "data" / "id_list.csv"

candidate_ids = pd.read_csv(
    file_path,
    dtype={"tmdb_id": "string"},
    skip_blank_lines=False
)

print("Original candidate list loaded.")
print("Project root:", project_root)
print("Dataset shape:", candidate_ids.shape)

Original candidate list loaded.
Project root: c:\Users\MichaelBoakye\OneDrive - AmaliTech gGmbH\Documents\Data_Engineering_and_Analytics_Labs\StreamVault_Licensing_Analysis
Dataset shape: (127, 1)


In [6]:
# Display the first five records
display(candidate_ids.head())

# Inspect dataset structure
candidate_ids.info()

# Count missing values
display(candidate_ids.isna().sum())

# Count duplicate rows
print("Duplicate rows:", candidate_ids.duplicated().sum())

# Identify duplicate TMDb IDs
display(
    candidate_ids[
        candidate_ids.duplicated(
            subset=["tmdb_id"],
            keep=False
        )
    ]
)

,tmdb_id
0,155
1,165
2,242
3,49026
4,8077


<class 'pandas.DataFrame'>
RangeIndex: 127 entries, 0 to 126
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   tmdb_id  125 non-null    string
dtypes: string(1)
memory usage: 1.1 KB


tmdb_id    2
dtype: int64

Duplicate rows: 7


,tmdb_id
0,155
6,680
25,278
38,603
41,603
53,1891
58,278
67,<NA>
78,550
88,1891


#### 1.2 Identifying Invalid Identifiers

TMDb IDs should contain only positive numeric values. This step identifies non-numeric, zero and negative identifiers that cannot be used for movie extraction.

In [7]:
# Identify non-numeric identifiers
non_numeric_ids = candidate_ids[
    candidate_ids["tmdb_id"].notna()
    & ~candidate_ids["tmdb_id"].str.fullmatch(r"\d+")
]

print("Non-numeric identifiers:")
display(non_numeric_ids)

# Identify zero and negative numeric identifiers
numeric_ids = candidate_ids[
    candidate_ids["tmdb_id"].str.fullmatch(r"\d+").fillna(False)
]

invalid_numeric_ids = numeric_ids[
    numeric_ids["tmdb_id"].astype("Int64") <= 0
]

print("Zero or negative identifiers:")
display(invalid_numeric_ids)

# Summarise distinct non-missing identifiers
print(
    "Distinct non-missing IDs:",
    candidate_ids["tmdb_id"].nunique()
)

Non-numeric identifiers:


,tmdb_id
28,-1
93,tt0468569


Zero or negative identifiers:


,tmdb_id


Distinct non-missing IDs: 119


#### 1.3 Preparing the Extraction List

Remove missing, duplicate and invalid TMDb IDs to produce a clean list of unique extraction candidates. The original dataset remains unchanged.

In [8]:
extraction_ids = candidate_ids.dropna(
    subset=["tmdb_id"]
).drop_duplicates(
    subset=["tmdb_id"]
).copy()

extraction_ids = extraction_ids[
    extraction_ids["tmdb_id"].str.fullmatch(
        r"[1-9]\d*"
    ).fillna(False)
].copy()

extraction_ids = extraction_ids.reset_index(drop=True)

print("Valid unique extraction candidates:", len(extraction_ids))

assert extraction_ids["tmdb_id"].is_unique
assert extraction_ids["tmdb_id"].notna().all()

Valid unique extraction candidates: 117


### 2. TMDb API Exploration

Explore the TMDb API by testing movie details and credits endpoints and examining the information returned.

In [9]:
import os
import requests
from dotenv import load_dotenv

# Load API credentials
load_dotenv(project_root / ".env")

tmdb_api_key = os.getenv("TMDB_API_KEY")

if not tmdb_api_key:
    raise ValueError("TMDB_API_KEY not found in .env")

# Create API session
tmdb_session = requests.Session()
tmdb_session.params.update({"api_key": tmdb_api_key})

print("TMDb API key loaded:", bool(tmdb_api_key))

TMDb API key loaded: True


#### 2.1 Testing the Movie Details Endpoint

Retrieve a sample movie record to examine the details returned by TMDb.

In [10]:
# Select the first valid candidate
test_movie_id = str(extraction_ids["tmdb_id"].iloc[0])

# Request movie details
response = tmdb_session.get(
    f"https://api.themoviedb.org/3/movie/{test_movie_id}",
    timeout=30
)

response.raise_for_status()

movie_details = response.json()

print("Status code:", response.status_code)
print("Movie ID:", movie_details["id"])
print("Movie title:", movie_details["title"])

Status code: 200
Movie ID: 155
Movie title: The Dark Knight


#### 2.2 Testing the Movie Credits Endpoint

Retrieve the movie's credits to examine its cast and crew information, including directors and actors.

In [11]:
# Request movie credits
credits_response = tmdb_session.get(
    f"https://api.themoviedb.org/3/movie/{test_movie_id}/credits",
    timeout=30
)

credits_response.raise_for_status()

credits_data = credits_response.json()

# Extract directors
directors = [
    person["name"]
    for person in credits_data["crew"]
    if person["job"] == "Director"
]

# Display the first five cast members
print("Directors:", directors)

print("\nFirst five cast members:")
for person in credits_data["cast"][:5]:
    print(person["name"], "-", person["character"])

Directors: ['Christopher Nolan']

First five cast members:
Christian Bale - Bruce Wayne
Heath Ledger - Joker
Aaron Eckhart - Harvey Dent
Michael Caine - Alfred
Maggie Gyllenhaal - Rachel


#### 2.3 Inspecting the API Response Structure

Examine the fields returned by both endpoints to identify the information available for the licensing analysis.

In [12]:
print("MOVIE DETAILS FIELDS")
print(list(movie_details.keys()))

print("\nCREDITS FIELDS")
print(list(credits_data.keys()))

MOVIE DETAILS FIELDS
['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']

CREDITS FIELDS
['id', 'cast', 'crew']


#### 2.4 Examining Collection Information

Inspect the collection field to determine how TMDb identifies movies belonging to a franchise.

In [13]:
print("Movie:", movie_details["title"])
print("Collection:", movie_details["belongs_to_collection"])

Movie: The Dark Knight
Collection: {'id': 263, 'name': 'The Dark Knight Collection', 'poster_path': '/l4T8JVwircGZlZuSl29U3TS9mpl.jpg', 'backdrop_path': '/xyhrCEdB4XRkelfVsqXeUZ6rLHi.jpg'}


### 3. Data Loading and Structuring

Load the saved TMDb movie and credits responses and organise them into separate movie, cast and director datasets for analysis.

In [14]:
import json

# Locate saved raw API responses
raw_data_path = project_root / "data" / "raw"

print("Raw data directory:", raw_data_path)
print("Directory exists:", raw_data_path.exists())

# Inspect available files
raw_files = list(raw_data_path.glob("*.json"))

print("Number of saved JSON files:", len(raw_files))
print("Sample filenames:", [file.name for file in raw_files[:5]])

Raw data directory: c:\Users\MichaelBoakye\OneDrive - AmaliTech gGmbH\Documents\Data_Engineering_and_Analytics_Labs\StreamVault_Licensing_Analysis\data\raw
Directory exists: True
Number of saved JSON files: 224
Sample filenames: ['credits_10138.json', 'credits_10192.json', 'credits_10193.json', 'credits_105.json', 'credits_10681.json']


#### 3.1 Loading and Organising the Raw Data

Load the saved JSON responses into separate datasets for movies, cast members and directors. Keeping these datasets separate preserves their different levels of detail.

In [15]:
# Separate movie details and credits files
movie_files = list(raw_data_path.glob("movie_*.json"))
credits_files = list(raw_data_path.glob("credits_*.json"))

# Load movie details
movie_records = []

for file in movie_files:
    with open(file, "r", encoding="utf-8") as f:
        movie_records.append(json.load(f))

# Load credits
credits_records = []

for file in credits_files:
    with open(file, "r", encoding="utf-8") as f:
        credits_records.append(json.load(f))

print("Movie records loaded:", len(movie_records))
print("Credits records loaded:", len(credits_records))

Movie records loaded: 112
Credits records loaded: 112


#### 3.2 Creating the Movie, Cast and Director DataFrames

Convert the raw API responses into structured Pandas DataFrames. Movie-level information, cast members and directors are stored separately to avoid duplicating movie records.

In [16]:
# Create the movie DataFrame
movies = pd.json_normalize(movie_records)

# Extract cast and director records
cast_records = []
director_records = []

for credits in credits_records:
    movie_id = credits["id"]

    for person in credits["cast"]:
        cast_records.append({
            "movie_id": movie_id,
            "actor_id": person["id"],
            "actor_name": person["name"],
            "character": person["character"],
            "cast_order": person["order"]
        })

    for person in credits["crew"]:
        if person["job"] == "Director":
            director_records.append({
                "movie_id": movie_id,
                "director_id": person["id"],
                "director_name": person["name"]
            })

# Create separate DataFrames
cast = pd.DataFrame(cast_records)
directors = pd.DataFrame(director_records)

print("Movies:", movies.shape)
print("Cast:", cast.shape)
print("Directors:", directors.shape)

Movies: (112, 31)
Cast: (7823, 5)
Directors: (122, 3)


#### 3.3 Inspecting the Dataset Structures

Examine the columns and sample records in each DataFrame to understand how movie, cast and director information is organised.

In [17]:
print("MOVIES")
display(movies.head())
print("Columns:", movies.columns.tolist())

print("\nCAST")
display(cast.head())
print("Columns:", cast.columns.tolist())

print("\nDIRECTORS")
display(directors.head())
print("Columns:", directors.columns.tolist())

MOVIES


,adult,backdrop_path,budget,genres,homepage,id,imdb_id,origin_country,original_language,original_title,...,tagline,title,video,vote_average,vote_count,belongs_to_collection.id,belongs_to_collection.name,belongs_to_collection.poster_path,belongs_to_collection.backdrop_path,belongs_to_collection
0,False,/7lmBufEG7P7Y1HClYK3gCxYrkgS.jpg,200000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",https://www.marvel.com/movies/iron-man-2,10138,tt1228705,[US],en,Iron Man 2,...,"It's not the armor that makes the hero, but th...",Iron Man 2,False,6.856,23271,131292.0,Iron Man Collection,/fbeJ7f0aD4A112Bc1tnpzyn82xO.jpg,/rI8zOWkRQJdlAyQ6WJOSlYK6JxZ.jpg,NaN
1,False,/uzzTystB8lL0mRDII5Sfs5HxgkI.jpg,165000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 12, 'nam...",http://www.shrekforeverafter.com/,10192,tt0892791,[US],en,Shrek Forever After,...,It ain't ogre... til it's ogre.,Shrek Forever After,False,6.417,8245,2150.0,Shrek Collection,/g1Kcbi5zSelArzR5UYkxKvG6K2a.jpg,/lhsd1zCsq5UquvcNalmhuddV3tI.jpg,NaN
2,False,/wE5JGzujfvDPMIfFjJyrhXFjZLc.jpg,200000000,"[{'id': 16, 'name': 'Animation'}, {'id': 10751...",http://disney.go.com/toystory/,10193,tt0435761,[US],en,Toy Story 3,...,No toy gets left behind.,Toy Story 3,False,7.818,16477,10194.0,Toy Story Collection,/rki5qLuwb0xnnE9seehxO9TlLhW.jpg,/hApclyB9NEZEQujAVajzi5iWE4a.jpg,NaN
3,False,/5bzPWQ2dFUl2aZKkp7ILJVVkRed.jpg,19000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 35, '...",http://www.backtothefuture.com/movies/backtoth...,105,tt0088763,[US],en,Back to the Future,...,He was never in time for his classes... He was...,Back to the Future,False,8.332,22818,264.0,Back to the Future Collection,/5Xsu2o5IsZRuuxCEVZ9nVve21FP.jpg,/c9C9Pg2QctyjZHRmS0P8rZg1OTA.jpg,NaN
4,False,/nYs4ZwnJBK4AgljhvzwNz7fpr3E.jpg,180000000,"[{'id': 16, 'name': 'Animation'}, {'id': 10751...",https://www.pixar.com/feature-films/walle/,10681,tt0910970,[US],en,WALL·E,...,After 700 years of doing what he was built for...,WALL·E,False,8.129,21235,NaN,NaN,NaN,NaN,NaN


Columns: ['adult', 'backdrop_path', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count', 'belongs_to_collection.id', 'belongs_to_collection.name', 'belongs_to_collection.poster_path', 'belongs_to_collection.backdrop_path', 'belongs_to_collection']

CAST


,movie_id,actor_id,actor_name,character,cast_order
0,10138,3223,Robert Downey Jr.,Tony Stark,0
1,10138,12052,Gwyneth Paltrow,Pepper Potts,1
2,10138,1896,Don Cheadle,Lt. Col. James 'Rhodey' Rhodes,2
3,10138,1245,Scarlett Johansson,Natalie Rushman / Natasha Romanoff,3
4,10138,6807,Sam Rockwell,Justin Hammer,4


Columns: ['movie_id', 'actor_id', 'actor_name', 'character', 'cast_order']

DIRECTORS


,movie_id,director_id,director_name
0,10138,15277,Jon Favreau
1,10192,64151,Mike Mitchell
2,10193,8,Lee Unkrich
3,105,24,Robert Zemeckis
4,10681,7,Andrew Stanton


Columns: ['movie_id', 'director_id', 'director_name']


### 4. Data Quality Assessment

Assess the completeness, uniqueness and consistency of the structured datasets before using them for licensing analysis.

#### 4.1 Checking Missing Values

Identify missing values in the movie, cast and director datasets, focusing on fields required for the analysis.

In [18]:
# Check missing values in each dataset

print("MOVIES: Missing values")
display(movies.isna().sum().sort_values(ascending=False).head(15))

print("\nCAST: Missing values")
display(cast.isna().sum())

print("\nDIRECTORS: Missing values")
display(directors.isna().sum())

MOVIES: Missing values


belongs_to_collection                  112
belongs_to_collection.poster_path       16
belongs_to_collection.id                16
belongs_to_collection.name              16
belongs_to_collection.backdrop_path     16
backdrop_path                            0
adult                                    0
origin_country                           0
original_language                        0
original_title                           0
budget                                   0
genres                                   0
homepage                                 0
id                                       0
imdb_id                                  0
dtype: int64


CAST: Missing values


movie_id      0
actor_id      0
actor_name    0
character     0
cast_order    0
dtype: int64


DIRECTORS: Missing values


movie_id         0
director_id      0
director_name    0
dtype: int64

#### 4.2 Consolidated Data Quality Checks

Evaluate essential movie fields, duplicate records and relationships between the movie, cast and director datasets.

In [19]:
# Essential movie-field checks
essential_fields = ["id", "title", "release_date", "budget", "revenue"]

print("MISSING ESSENTIAL VALUES")
display(movies[essential_fields].isna().sum())

print("\nZERO BUDGETS:", (movies["budget"] == 0).sum())
print("ZERO REVENUES:", (movies["revenue"] == 0).sum())

# Duplicate movie IDs
print("\nDUPLICATE MOVIE IDs:", movies["id"].duplicated().sum())

# Check relationships between datasets
movie_ids = set(movies["id"])
cast_movie_ids = set(cast["movie_id"])
director_movie_ids = set(directors["movie_id"])

print("\nRELATIONSHIP CHECKS")
print("Movies without cast records:", len(movie_ids - cast_movie_ids))
print("Movies without director records:", len(movie_ids - director_movie_ids))
print("Unmatched cast records:", len(cast_movie_ids - movie_ids))
print("Unmatched director records:", len(director_movie_ids - movie_ids))

MISSING ESSENTIAL VALUES


id              0
title           0
release_date    0
budget          0
revenue         0
dtype: int64


ZERO BUDGETS: 0
ZERO REVENUES: 0

DUPLICATE MOVIE IDs: 0

RELATIONSHIP CHECKS
Movies without cast records: 0
Movies without director records: 0
Unmatched cast records: 0
Unmatched director records: 0


### 5. Franchise and Financial Analysis

Classify movies using TMDb collection membership and compare the historical financial performance of franchise and standalone films.

In [20]:
# Create franchise classification
movies["is_franchise"] = movies["belongs_to_collection.id"].notna()

# Calculate gross-to-budget ratio
movies["gross_to_budget"] = (
    movies["revenue"] / movies["budget"]
)

# Extract release year
movies["release_year"] = pd.to_datetime(
    movies["release_date"],
    errors="coerce"
).dt.year

# Summarise franchise and standalone records
print("FRANCHISE CLASSIFICATION")
display(
    movies["is_franchise"]
    .value_counts()
    .rename(index={True: "Franchise", False: "Standalone"})
)

print("\nCOLLECTION COUNT:")
print(movies["belongs_to_collection.id"].nunique())

print("\nSAMPLE CLASSIFIED MOVIES:")
display(
    movies[
        ["title", "release_year", "belongs_to_collection.name",
         "is_franchise", "budget", "revenue", "gross_to_budget"]
    ].head(10)
)

FRANCHISE CLASSIFICATION


is_franchise
Franchise     96
Standalone    16
Name: count, dtype: int64


COLLECTION COUNT:
32

SAMPLE CLASSIFIED MOVIES:


,title,release_year,belongs_to_collection.name,is_franchise,budget,revenue,gross_to_budget
0,Iron Man 2,2010,Iron Man Collection,True,200000000,623933331,3.119667
1,Shrek Forever After,2010,Shrek Collection,True,165000000,752600867,4.561217
2,Toy Story 3,2010,Toy Story Collection,True,200000000,1067316101,5.336581
3,Back to the Future,1985,Back to the Future Collection,True,19000000,391546852,20.607729
4,WALL·E,2008,NaN,False,180000000,521311860,2.896177
5,Star Wars,1977,Star Wars Collection,True,11000000,775398007,70.490728
6,Scarface,1983,NaN,False,25000000,66023329,2.640933
7,The Prestige,2006,NaN,False,40000000,109676311,2.741908
8,The Lord of the Rings: The Fellowship of the Ring,2001,The Lord of the Rings Collection,True,93000000,871368364,9.369552
9,The Lord of the Rings: The Two Towers,2002,The Lord of the Rings Collection,True,79000000,926287400,11.725157


#### 5.1 Franchise vs Standalone Performance

Compare franchise and standalone movies using median revenue, median budget and gross-to-budget ratio. Repeat the comparison for all years and movies released from 2000 onward.

In [21]:
# Compare financial performance across both periods

def compare_performance(data, period):
    comparison = (
        data.groupby("is_franchise")
        .agg(
            movie_count=("id", "count"),
            median_revenue=("revenue", "median"),
            median_budget=("budget", "median"),
            median_gross_to_budget=("gross_to_budget", "median")
        )
        .rename(index={True: "Franchise", False: "Standalone"})
    )

    comparison.index.name = "Category"

    print(f"\n{period}")
    display(comparison)


# All available years
compare_performance(movies, "ALL YEARS")

# Movies released from 2000 onward
recent_movies = movies[movies["release_year"] >= 2000]

compare_performance(recent_movies, "2000 ONWARD")


ALL YEARS


,movie_count,median_revenue,median_budget,median_gross_to_budget
Category,,,,
Standalone,16,161802536.5,25000000.0,3.844812
Franchise,96,578937111.0,101000000.0,5.623799



2000 ONWARD


,movie_count,median_revenue,median_budget,median_gross_to_budget
Category,,,,
Standalone,8,283195047.5,65000000.0,3.844812
Franchise,63,786636033.0,150000000.0,5.051939
